<a href="https://colab.research.google.com/github/JoelmPradar/Calculadora_de_derivadas/blob/main/Copia_de_Codigo_CalculoDiferencial_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ipywidgets

from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
# ============================================================
# CELDA 1
# INSTALACIÓN, IMPORTS Y CONFIGURACIÓN GENERAL
# ============================================================

!pip -q install sympy ipywidgets plotly

import sympy as sp
import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
import plotly.io as pio

from IPython.display import display, Math, Markdown, HTML, clear_output
from sympy.calculus.util import continuous_domain
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication_application,
    convert_xor,
    function_exponentiation
)

# Activar widgets en Google Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except:
    pass

# Configuración de Plotly para Colab
try:
    pio.renderers.default = "colab"
except:
    pass


# ============================================================
# VARIABLE PRINCIPAL
# ============================================================

x = sp.symbols("x")


# ============================================================
# FUNCIONES PERMITIDAS PARA ESCRIBIR EN LA ENTRADA
# ============================================================

transformations = standard_transformations + (
    implicit_multiplication_application,
    convert_xor,
    function_exponentiation
)

funciones_permitidas = {
    "x": x,
    "pi": sp.pi,
    "e": sp.E,
    "E": sp.E,

    # Trigonométricas
    "sin": sp.sin,
    "sen": sp.sin,
    "cos": sp.cos,
    "tan": sp.tan,
    "sec": sp.sec,
    "csc": sp.csc,
    "cot": sp.cot,

    # Trigonométricas inversas
    "asin": sp.asin,
    "acos": sp.acos,
    "atan": sp.atan,
    "arcsin": sp.asin,
    "arccos": sp.acos,
    "arctan": sp.atan,

    # Exponencial, logaritmo y raíz
    "exp": sp.exp,
    "sqrt": sp.sqrt,
    "log": sp.log,
    "ln": sp.log,

    # Valor absoluto
    "abs": sp.Abs,
    "Abs": sp.Abs,
}

print("Celda 1 ejecutada correctamente.")

Celda 1 ejecutada correctamente.


In [ ]:
# ============================================================
# CELDA 2
# LECTURA DE FUNCIÓN, DERIVADAS Y FUNCIONES NUMÉRICAS
# ============================================================

def leer_funcion(cadena):
    """
    Convierte el texto escrito por el usuario en una expresión simbólica.
    La función debe estar escrita solamente en términos de x.
    """

    expr = parse_expr(
        cadena,
        local_dict=funciones_permitidas,
        transformations=transformations,
        evaluate=True
    )

    simbolos = expr.free_symbols

    if simbolos - {x}:
        raise ValueError(
            f"La función solo debe tener la variable x. "
            f"Encontré estas otras variables: {simbolos - {x}}"
        )

    return expr


def derivada_no_simplificada(expr, var=x):
    """
    Calcula una derivada intentando conservar una forma no simplificada.
    Sirve para mostrar la expresión antes de simplificar.
    """

    expr = sp.sympify(expr)

    # Constante
    if expr.is_Number:
        return sp.Integer(0)

    # Variable principal
    if expr == var:
        return sp.Integer(1)

    # Otro símbolo
    if expr.is_Symbol:
        return sp.Integer(0)

    # Suma o resta
    if isinstance(expr, sp.Add):
        return sp.Add(
            *[derivada_no_simplificada(t, var) for t in expr.as_ordered_terms()],
            evaluate=False
        )

    # Producto o cociente
    if isinstance(expr, sp.Mul):
        numerador, denominador = sp.fraction(expr)

        # Si hay denominador variable, se aplica regla del cociente
        if denominador != 1 and denominador.has(var):
            u = numerador
            v = denominador

            u_der = derivada_no_simplificada(u, var)
            v_der = derivada_no_simplificada(v, var)

            return sp.Mul(
                sp.Add(
                    sp.Mul(u_der, v, evaluate=False),
                    sp.Mul(-1, u, v_der, evaluate=False),
                    evaluate=False
                ),
                sp.Pow(v, -2, evaluate=False),
                evaluate=False
            )

        coeff, rest = expr.as_coeff_Mul()

        # Constante por función
        if coeff != 1:
            return sp.Mul(
                coeff,
                derivada_no_simplificada(rest, var),
                evaluate=False
            )

        # Producto general
        factors = expr.as_ordered_factors()
        partes = []

        for i, factor in enumerate(factors):
            derivada_factor = derivada_no_simplificada(factor, var)
            producto = []

            for j, otro_factor in enumerate(factors):
                if i == j:
                    producto.append(derivada_factor)
                else:
                    producto.append(otro_factor)

            partes.append(sp.Mul(*producto, evaluate=False))

        return sp.Add(*partes, evaluate=False)

    # Potencias
    if isinstance(expr, sp.Pow):
        base, exponente = expr.as_base_exp()

        # Caso u^n
        if not exponente.has(var):
            return sp.Mul(
                exponente,
                sp.Pow(base, exponente - 1, evaluate=False),
                derivada_no_simplificada(base, var),
                evaluate=False
            )

        # Caso a^v
        if not base.has(var) and exponente.has(var):
            return sp.Mul(
                expr,
                sp.log(base),
                derivada_no_simplificada(exponente, var),
                evaluate=False
            )

        # Caso u^v
        u = base
        v = exponente

        u_der = derivada_no_simplificada(u, var)
        v_der = derivada_no_simplificada(v, var)

        return sp.Mul(
            sp.Pow(u, v, evaluate=False),
            sp.Add(
                sp.Mul(v_der, sp.log(u), evaluate=False),
                sp.Mul(v, u_der, 1/u, evaluate=False),
                evaluate=False
            ),
            evaluate=False
        )

    # Funciones compuestas
    if expr.is_Function:
        nombre = expr.func.__name__
        arg = expr.args[0]
        darg = derivada_no_simplificada(arg, var)

        if nombre == "sin":
            return sp.Mul(sp.cos(arg), darg, evaluate=False)

        if nombre == "cos":
            return sp.Mul(-1, sp.sin(arg), darg, evaluate=False)

        if nombre == "tan":
            return sp.Mul(sp.sec(arg)**2, darg, evaluate=False)

        if nombre == "sec":
            return sp.Mul(sp.sec(arg), sp.tan(arg), darg, evaluate=False)

        if nombre == "csc":
            return sp.Mul(-1, sp.csc(arg), sp.cot(arg), darg, evaluate=False)

        if nombre == "cot":
            return sp.Mul(-1, sp.csc(arg)**2, darg, evaluate=False)

        if nombre == "exp":
            return sp.Mul(sp.exp(arg), darg, evaluate=False)

        if nombre == "log":
            return sp.Mul(darg, 1 / arg, evaluate=False)

        if nombre == "asin":
            return sp.Mul(darg, 1 / sp.sqrt(1 - arg**2), evaluate=False)

        if nombre == "acos":
            return sp.Mul(-1, darg, 1 / sp.sqrt(1 - arg**2), evaluate=False)

        if nombre == "atan":
            return sp.Mul(darg, 1 / (1 + arg**2), evaluate=False)

        if nombre == "Abs":
            return sp.Mul(arg / sp.Abs(arg), darg, evaluate=False)

    # Respaldo simbólico
    return sp.diff(expr, var)


def convertir_a_funcion_numerica(expr):
    """
    Convierte una expresión simbólica a una función numérica para graficar.
    """

    expr = sp.simplify(expr)
    expr = expr.rewrite(sp.sin).rewrite(sp.cos)

    funciones_numpy_extra = {
        "sec": lambda z: 1 / np.cos(z),
        "csc": lambda z: 1 / np.sin(z),
        "cot": lambda z: 1 / np.tan(z),
        "Abs": np.abs,
    }

    return sp.lambdify(x, expr, modules=[funciones_numpy_extra, "numpy"])


def evaluar_en_reales(funcion_num, valores_x):
    """
    Evalúa una función y elimina valores complejos, infinitos o indefinidos.
    """

    with np.errstate(all="ignore"):
        y = funcion_num(valores_x)

    y = np.asarray(y, dtype=np.complex128)

    if y.shape == ():
        y = np.full_like(valores_x, y, dtype=np.complex128)

    # Eliminar valores complejos
    y[np.abs(y.imag) > 1e-8] = np.nan + 1j*np.nan

    y_real = y.real.astype(float)

    # Eliminar infinitos o indefinidos
    y_real[~np.isfinite(y_real)] = np.nan

    # Evitar escalas exageradas por asíntotas
    y_real[np.abs(y_real) > 1e6] = np.nan

    return y_real


print("Celda 2 ejecutada correctamente.")

Celda 2 ejecutada correctamente.


In [ ]:
# ============================================================
# CELDA 3
# RAÍCES, CEROS Y ANÁLISIS DE DOMINIO
# ============================================================

def es_real_aproximado(valor):
    """
    Verifica si una raíz es real usando aproximación numérica.
    """

    try:
        c = complex(sp.N(valor, 15))
        return abs(c.imag) < 1e-10
    except:
        return False


def ordenar_raices(raices):
    """
    Ordena raíces reales si es posible.
    """

    try:
        return sorted(raices, key=lambda r: float(sp.N(r)))
    except:
        return raices


def eliminar_duplicados_numericos(lista, tolerancia=1e-6):
    """
    Elimina raíces numéricas repetidas.
    """

    resultado = []

    for valor in lista:
        if not np.isfinite(valor):
            continue

        repetido = False

        for r in resultado:
            if abs(valor - r) < tolerancia:
                repetido = True
                break

        if not repetido:
            resultado.append(valor)

    return resultado


def busqueda_numerica_raices(expr, xmin, xmax):
    """
    Busca raíces numéricas dentro del intervalo [xmin, xmax].
    """

    f_num = convertir_a_funcion_numerica(expr)

    xs = np.linspace(xmin, xmax, 2500)
    ys = evaluar_en_reales(f_num, xs)

    raices = []

    for i in range(len(xs) - 1):
        x1, x2 = xs[i], xs[i + 1]
        y1, y2 = ys[i], ys[i + 1]

        if np.isnan(y1) or np.isnan(y2):
            continue

        if abs(y1) < 1e-5:
            raices.append(float(x1))

        if y1 * y2 < 0:
            try:
                raiz = sp.nsolve(expr, x, (x1, x2))
                raiz = float(raiz)

                if xmin <= raiz <= xmax:
                    raices.append(raiz)

            except:
                try:
                    raiz = sp.nsolve(expr, x, (x1 + x2) / 2)
                    raiz = float(raiz)

                    if xmin <= raiz <= xmax:
                        raices.append(raiz)

                except:
                    pass

    return eliminar_duplicados_numericos(raices)


def mostrar_raices_y_ceros(expr, xmin, xmax):
    """
    Muestra raíces o ceros de la función con procedimiento.
    """

    display(Markdown("### Raíces / ceros de la función"))

    display(Markdown("**Paso 1. Igualar la función a cero**"))
    display(Markdown("**Regla usada:** Para encontrar raíces se resuelve `f(x)=0`."))
    display(Math(sp.latex(sp.Eq(expr, 0, evaluate=False))))

    expr_simplificada = sp.simplify(expr)

    display(Markdown("**Paso 2. Simplificar la expresión si es posible**"))
    display(Markdown("**Regla usada:** Simplificación algebraica."))
    display(Math(r"f(x)=" + sp.latex(expr_simplificada)))

    factor_expr = sp.factor(expr_simplificada)

    display(Markdown("**Paso 3. Factorizar la función si es posible**"))
    display(Markdown("**Regla usada:** Factorización algebraica."))

    if factor_expr != expr_simplificada:
        display(Math(sp.latex(sp.Eq(expr_simplificada, factor_expr, evaluate=False))))
    else:
        display(Math(r"\text{No se pudo factorizar más.}"))

    try:
        sp.Poly(expr_simplificada, x)
        es_polinomio = True
    except:
        es_polinomio = False

    if es_polinomio:
        display(Markdown("**Paso 4. Aplicar propiedad del producto cero**"))
        display(Markdown(
            "**Regla usada:** Si un producto es igual a cero, entonces al menos uno de sus factores debe ser cero."
        ))

        try:
            coef, factores = sp.factor_list(expr_simplificada)

            if len(factores) > 0:
                for factor, potencia in factores:
                    display(Math(rf"{sp.latex(factor)}=0"))

                    soluciones_factor = sp.solve(sp.Eq(factor, 0), x)
                    soluciones_reales = [
                        r for r in soluciones_factor
                        if es_real_aproximado(r)
                    ]

                    if len(soluciones_reales) > 0:
                        display(Math(
                            r"x="
                            + r"\quad \text{o} \quad x=".join(
                                [sp.latex(r) for r in soluciones_reales]
                            )
                        ))
                    else:
                        display(Math(r"\text{Este factor no tiene raíces reales.}"))
            else:
                display(Math(r"\text{La función no tiene factores variables.}"))

        except:
            display(Math(r"\text{No se pudo descomponer por factores.}"))

    else:
        display(Markdown("**Paso 4. Resolver la ecuación en los números reales**"))
        display(Markdown(
            "**Regla usada:** Si no es polinómica, se intenta resolver simbólica y numéricamente."
        ))

    display(Markdown("**Resultado de raíces reales encontradas:**"))

    try:
        solucion_exacta = sp.solveset(expr_simplificada, x, domain=sp.S.Reals)

        if solucion_exacta == sp.EmptySet:
            display(Math(r"\text{No tiene raíces reales exactas.}"))

        elif isinstance(solucion_exacta, sp.FiniteSet):
            raices = ordenar_raices(list(solucion_exacta))

            if len(raices) == 0:
                display(Math(r"\text{No tiene raíces reales.}"))
            else:
                display(Math(
                    r"x="
                    + r"\quad \text{o} \quad x=".join(
                        [sp.latex(r) for r in raices]
                    )
                ))

                display(Markdown("**Aproximaciones decimales:**"))
                display(Math(
                    r"x\approx "
                    + r"\quad \text{o} \quad x\approx ".join(
                        [sp.latex(sp.N(r, 10)) for r in raices]
                    )
                ))

        else:
            display(Math(
                r"\text{Solución simbólica obtenida: }"
                + sp.latex(solucion_exacta)
            ))

            raices_numericas = busqueda_numerica_raices(expr_simplificada, xmin, xmax)

            if len(raices_numericas) > 0:
                display(Markdown(f"**Raíces numéricas dentro del dominio [{xmin}, {xmax}]:**"))
                display(Math(
                    r"x\approx "
                    + r"\quad \text{o} \quad x\approx ".join(
                        [sp.latex(round(r, 8)) for r in raices_numericas]
                    )
                ))

    except:
        raices_numericas = busqueda_numerica_raices(expr_simplificada, xmin, xmax)

        if len(raices_numericas) > 0:
            display(Markdown(f"**Raíces numéricas dentro del dominio [{xmin}, {xmax}]:**"))
            display(Math(
                r"x\approx "
                + r"\quad \text{o} \quad x\approx ".join(
                    [sp.latex(round(r, 8)) for r in raices_numericas]
                )
            ))
        else:
            display(Math(r"\text{No se encontraron raíces reales en el dominio elegido.}"))


def obtener_dominio(expr):
    """
    Intenta obtener el dominio real de una expresión.
    """

    try:
        return continuous_domain(expr, x, sp.S.Reals)
    except:
        return None


def intervalo_recomendado_desde_dominio(dominio):
    """
    Intenta proponer un intervalo numérico útil para graficar.
    """

    if dominio is None:
        return None

    if dominio == sp.S.Reals:
        return (-5, 5)

    intervalos = []

    if isinstance(dominio, sp.Interval):
        intervalos = [dominio]

    elif isinstance(dominio, sp.Union):
        intervalos = [
            arg for arg in dominio.args
            if isinstance(arg, sp.Interval)
        ]

    if len(intervalos) == 0:
        return None

    elegido = None

    for I in intervalos:
        try:
            if bool(I.contains(0)):
                elegido = I
                break
        except:
            pass

    if elegido is None:
        elegido = intervalos[0]

    a = elegido.start
    b = elegido.end

    try:
        if a == -sp.oo and b == sp.oo:
            left, right = -5, 5

        elif a == -sp.oo:
            right = float(sp.N(b))
            left = right - 10

        elif b == sp.oo:
            left = float(sp.N(a))
            right = left + 10

        else:
            left = float(sp.N(a))
            right = float(sp.N(b))

        ancho = right - left

        if ancho <= 0 or not np.isfinite(ancho):
            return None

        margen = max(0.02 * ancho, 0.001)

        if elegido.left_open:
            left += margen

        if elegido.right_open:
            right -= margen

        if left >= right:
            return None

        return (round(left, 5), round(right, 5))

    except:
        return None


def mostrar_analisis_dominio(expr, derivada, segunda_derivada, xmin, xmax):
    """
    Muestra dominio de f(x), f'(x), f''(x) y recomienda intervalo.
    """

    display(Markdown("## 5. Análisis de dominio y recomendación antes de graficar"))

    dominio_f = obtener_dominio(expr)
    dominio_fp = obtener_dominio(derivada)
    dominio_fpp = obtener_dominio(segunda_derivada)

    display(Markdown("### Dominio de la función original"))

    if dominio_f is None:
        display(Math(r"\text{No se pudo determinar simbólicamente el dominio exacto de } f(x)."))
    else:
        display(Math(r"D_f=" + sp.latex(dominio_f)))

        if dominio_f == sp.S.Reals:
            display(Markdown(
                "**La función existe y es continua para todo número real.** "
                "Puedes usar cualquier intervalo real para graficarla."
            ))
        else:
            display(Markdown(
                "**La función no existe o no es continua en todos los reales.** "
                "Conviene escoger un intervalo que esté completamente dentro del dominio."
            ))

    display(Markdown("### Dominio de la derivada y de la segunda derivada"))

    if dominio_fp is not None:
        display(Math(r"D_{f'}=" + sp.latex(dominio_fp)))
    else:
        display(Math(r"\text{No se pudo determinar simbólicamente }D_{f'}."))

    if dominio_fpp is not None:
        display(Math(r"D_{f''}=" + sp.latex(dominio_fpp)))
    else:
        display(Math(r"\text{No se pudo determinar simbólicamente }D_{f''}."))

    dominio_comun = None

    try:
        if dominio_f is not None and dominio_fp is not None and dominio_fpp is not None:
            dominio_comun = dominio_f.intersect(dominio_fp).intersect(dominio_fpp)

            display(Markdown("### Dominio común recomendado para la gráfica con tangente"))
            display(Math(r"D_{\text{común}}=" + sp.latex(dominio_comun)))
    except:
        dominio_comun = None

    try:
        intervalo_usuario = sp.Interval(float(xmin), float(xmax))

        if dominio_f is not None:
            pertenece = intervalo_usuario.is_subset(dominio_f)

            if pertenece is True:
                display(Markdown(
                    f"El intervalo que escribiste, **[{xmin}, {xmax}]**, está dentro del dominio de la función."
                ))
            elif pertenece is False:
                display(Markdown(
                    f"⚠️ El intervalo que escribiste, **[{xmin}, {xmax}]**, contiene valores fuera del dominio. "
                    "La gráfica puede mostrar huecos, cortes, asíntotas o valores indefinidos."
                ))
    except:
        pass

    dominio_para_recomendar = dominio_comun if dominio_comun not in [None, sp.EmptySet] else dominio_f
    recomendado = intervalo_recomendado_desde_dominio(dominio_para_recomendar)

    display(Markdown("### Recomendación práctica"))

    if recomendado is not None:
        display(Markdown(
            f"Un intervalo razonable para probar la gráfica es: "
            f"**x mínimo = {recomendado[0]}**, **x máximo = {recomendado[1]}**."
        ))
    else:
        display(Markdown(
            "No se pudo proponer un intervalo numérico automático. "
            "Escoge un intervalo que no cruce discontinuidades, asíntotas o puntos fuera del dominio."
        ))


print("Celda 3 ejecutada correctamente.")

Celda 3 ejecutada correctamente.


In [ ]:
# ============================================================
# CELDA 4
# PROCEDIMIENTO ROBUSTO Y GRÁFICA INTERACTIVA
# ============================================================

def mostrar_procedimiento_derivada_robusto(expr, var=x):
    """
    Muestra un procedimiento recursivo.
    Descompone sumas, productos, cocientes, potencias y funciones compuestas.
    """

    contador = {"paso": 1}

    def nuevo_paso(titulo, regla, expresion_latex):
        display(Markdown(f"**Paso {contador['paso']}. {titulo}**"))
        display(Markdown(f"**Regla usada:** {regla}"))
        display(Math(expresion_latex))
        contador["paso"] += 1

    def explicar(e, nivel=0, max_nivel=10):
        e = sp.sympify(e)

        if nivel > max_nivel:
            nuevo_paso(
                "Expresión muy profunda",
                "Se continúa aplicando reglas de derivación de forma recursiva",
                rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
                rf"={sp.latex(derivada_no_simplificada(e, var))}"
            )
            return

        if e.is_Number:
            nuevo_paso(
                "Derivar una constante",
                "La derivada de una constante es cero",
                rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)=0"
            )
            return

        if e == var:
            nuevo_paso(
                "Derivar la variable principal",
                "La derivada de x respecto a x es 1",
                r"\frac{d}{dx}(x)=1"
            )
            return

        if e.is_Symbol:
            nuevo_paso(
                "Derivar un símbolo constante",
                "Todo símbolo diferente de x se toma como constante",
                rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)=0"
            )
            return

        if isinstance(e, sp.Add):
            terms = e.as_ordered_terms()

            nuevo_paso(
                "Separar la suma o resta",
                "Linealidad: la derivada de una suma es la suma de las derivadas",
                rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)="
                + " + ".join(
                    [rf"\frac{{d}}{{dx}}\left({sp.latex(t)}\right)" for t in terms]
                )
            )

            for t in terms:
                explicar(t, nivel + 1, max_nivel)

            nuevo_paso(
                "Unir los resultados",
                "Simplificación algebraica",
                rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
                rf"={sp.latex(sp.factor(sp.simplify(sp.diff(e, var))))}"
            )
            return

        if isinstance(e, sp.Mul):
            numerador, denominador = sp.fraction(e)

            if denominador != 1 and denominador.has(var):
                u = numerador
                v = denominador

                nuevo_paso(
                    "Identificar un cociente",
                    "Regla del cociente: (u/v)' = (u'v - uv') / v²",
                    rf"u={sp.latex(u)},\quad v={sp.latex(v)}"
                )

                explicar(u, nivel + 1, max_nivel)
                explicar(v, nivel + 1, max_nivel)

                nuevo_paso(
                    "Aplicar regla del cociente",
                    "Sustitución en la regla del cociente",
                    rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)="
                    rf"\frac{{\left({sp.latex(sp.diff(u, var))}\right)\left({sp.latex(v)}\right)"
                    rf"-\left({sp.latex(u)}\right)\left({sp.latex(sp.diff(v, var))}\right)}}"
                    rf"{{\left({sp.latex(v)}\right)^2}}"
                )

                nuevo_paso(
                    "Simplificar el cociente",
                    "Simplificación algebraica",
                    rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
                    rf"={sp.latex(sp.factor(sp.simplify(sp.diff(e, var))))}"
                )
                return

            coeff, rest = e.as_coeff_Mul()

            if coeff != 1:
                nuevo_paso(
                    "Sacar la constante multiplicando",
                    "Regla del factor constante: (c f)' = c f'",
                    rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)="
                    rf"{sp.latex(coeff)}\frac{{d}}{{dx}}\left({sp.latex(rest)}\right)"
                )

                explicar(rest, nivel + 1, max_nivel)

                nuevo_paso(
                    "Multiplicar por la constante",
                    "Simplificación algebraica",
                    rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
                    rf"={sp.latex(sp.factor(sp.simplify(sp.diff(e, var))))}"
                )
                return

            factors = e.as_ordered_factors()

            if len(factors) >= 2:
                nuevo_paso(
                    "Identificar un producto de factores",
                    "Regla del producto general",
                    rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
                )

                for i, factor in enumerate(factors, start=1):
                    display(Markdown(f"Factor {i}:"))
                    display(Math(sp.latex(factor)))
                    explicar(factor, nivel + 1, max_nivel)

                suma_productos = derivada_no_simplificada(e, var)

                nuevo_paso(
                    "Aplicar regla del producto",
                    "Se deriva un factor a la vez y los demás se conservan",
                    rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
                    rf"={sp.latex(suma_productos)}"
                )

                nuevo_paso(
                    "Simplificar el producto",
                    "Simplificación algebraica",
                    rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
                    rf"={sp.latex(sp.factor(sp.simplify(sp.diff(e, var))))}"
                )
                return

        if isinstance(e, sp.Pow):
            base, exponente = e.as_base_exp()

            if not exponente.has(var):
                nuevo_paso(
                    "Identificar una potencia de exponente constante",
                    "Regla de la potencia con cadena: (u^n)' = n u^(n-1) u'",
                    rf"u={sp.latex(base)},\quad n={sp.latex(exponente)}"
                )

                if base != var:
                    explicar(base, nivel + 1, max_nivel)

                nuevo_paso(
                    "Aplicar regla de la potencia",
                    "Sustitución en la regla de potencia",
                    rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)="
                    rf"{sp.latex(exponente)}"
                    rf"\left({sp.latex(base)}\right)^{{{sp.latex(exponente - 1)}}}"
                    rf"\left({sp.latex(sp.diff(base, var))}\right)"
                )

                nuevo_paso(
                    "Simplificar potencia derivada",
                    "Simplificación algebraica",
                    rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
                    rf"={sp.latex(sp.factor(sp.simplify(sp.diff(e, var))))}"
                )
                return

            if not base.has(var) and exponente.has(var):
                nuevo_paso(
                    "Identificar exponencial de base constante",
                    "Regla: (a^v)' = a^v ln(a) v'",
                    rf"a={sp.latex(base)},\quad v={sp.latex(exponente)}"
                )

                explicar(exponente, nivel + 1, max_nivel)

                nuevo_paso(
                    "Aplicar regla de exponencial de base constante",
                    "Sustitución en la regla exponencial",
                    rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)="
                    rf"{sp.latex(e)}\ln\left({sp.latex(base)}\right)"
                    rf"\left({sp.latex(sp.diff(exponente, var))}\right)"
                )

                nuevo_paso(
                    "Simplificar",
                    "Simplificación algebraica",
                    rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
                    rf"={sp.latex(sp.factor(sp.simplify(sp.diff(e, var))))}"
                )
                return

            nuevo_paso(
                "Identificar potencia con base y exponente variables",
                "Derivación logarítmica: (u^v)' = u^v [v' ln(u) + v u'/u]",
                rf"u={sp.latex(base)},\quad v={sp.latex(exponente)}"
            )

            explicar(base, nivel + 1, max_nivel)
            explicar(exponente, nivel + 1, max_nivel)

            nuevo_paso(
                "Aplicar derivación logarítmica",
                "Sustitución en la regla para u^v",
                rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)="
                rf"{sp.latex(base)}^{{{sp.latex(exponente)}}}"
                rf"\left["
                rf"\left({sp.latex(sp.diff(exponente, var))}\right)\ln\left({sp.latex(base)}\right)"
                rf"+{sp.latex(exponente)}\frac{{{sp.latex(sp.diff(base, var))}}}{{{sp.latex(base)}}}"
                rf"\right]"
            )

            nuevo_paso(
                "Simplificar",
                "Simplificación algebraica",
                rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
                rf"={sp.latex(sp.factor(sp.simplify(sp.diff(e, var))))}"
            )
            return

        if e.is_Function:
            nombre = e.func.__name__
            arg = e.args[0]

            reglas = {
                "sin": r"(\sin u)'=\cos(u)u'",
                "cos": r"(\cos u)'=-\sin(u)u'",
                "tan": r"(\tan u)'=\sec^2(u)u'",
                "sec": r"(\sec u)'=\sec(u)\tan(u)u'",
                "csc": r"(\csc u)'=-\csc(u)\cot(u)u'",
                "cot": r"(\cot u)'=-\csc^2(u)u'",
                "exp": r"(e^u)'=e^u u'",
                "log": r"(\ln u)'=\frac{u'}{u}",
                "asin": r"(\arcsin u)'=\frac{u'}{\sqrt{1-u^2}}",
                "acos": r"(\arccos u)'=-\frac{u'}{\sqrt{1-u^2}}",
                "atan": r"(\arctan u)'=\frac{u'}{1+u^2}",
                "Abs": r"(|u|)'=\frac{u}{|u|}u',\quad u\neq0",
            }

            regla = reglas.get(nombre, r"\text{Regla de la cadena general}")

            nuevo_paso(
                "Identificar función compuesta",
                f"Regla de la cadena. Se usa: ${regla}$",
                rf"u={sp.latex(arg)},\quad u'=\frac{{d}}{{dx}}\left({sp.latex(arg)}\right)"
            )

            if arg != var:
                explicar(arg, nivel + 1, max_nivel)

            nuevo_paso(
                "Aplicar la regla correspondiente",
                "Sustitución en la regla de la función externa",
                rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
                rf"={sp.latex(derivada_no_simplificada(e, var))}"
            )

            nuevo_paso(
                "Simplificar",
                "Simplificación algebraica",
                rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
                rf"={sp.latex(sp.factor(sp.simplify(sp.diff(e, var))))}"
            )
            return

        nuevo_paso(
            "Reescribir expresión en forma derivable",
            "Se aplican reglas elementales equivalentes",
            rf"\frac{{d}}{{dx}}\left({sp.latex(e)}\right)"
            rf"={sp.latex(sp.factor(sp.simplify(sp.diff(e, var))))}"
        )

    explicar(expr)


def formato_numero(valor, decimales=6):
    """
    Formatea números para mostrarlos en la gráfica.
    """

    try:
        v = float(sp.N(valor))
        if abs(v) < 1e-10:
            v = 0
        return f"{v:.{decimales}g}"
    except:
        return str(valor)


def texto_ecuacion_punto_pendiente(a, y0, m):
    """
    Construye el texto de la ecuación punto-pendiente.
    """

    b = y0 - m * a

    return (
        f"<b>Tangente a f'(x)</b><br>"
        f"Punto de tangencia: ({formato_numero(a)}, {formato_numero(y0)})<br>"
        f"Pendiente: m = f''(a) = {formato_numero(m)}<br><br>"
        f"<b>Ecuación punto-pendiente:</b><br>"
        f"y - ({formato_numero(y0)}) = ({formato_numero(m)})(x - ({formato_numero(a)}))<br><br>"
        f"<b>Forma despejada:</b><br>"
        f"y = ({formato_numero(m)})x + ({formato_numero(b)})"
    )


def grafica_interactiva(expr, derivada, segunda_derivada, xmin, xmax):
    """
    Gráfica interactiva con:
    f(x), f'(x), tangente a f'(x) y ecuación punto-pendiente.
    """

    valores_x = np.linspace(xmin, xmax, 1600)

    f_num = convertir_a_funcion_numerica(expr)
    d_num = convertir_a_funcion_numerica(derivada)

    y_f = evaluar_en_reales(f_num, valores_x)
    y_d = evaluar_en_reales(d_num, valores_x)

    valores_a = np.linspace(xmin, xmax, 61)

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=valores_x,
        y=y_f,
        mode="lines",
        name="f(x)",
        visible=True
    ))

    fig.add_trace(go.Scatter(
        x=valores_x,
        y=y_d,
        mode="lines",
        name="f'(x)",
        visible=True
    ))

    anotaciones = []

    for a in valores_a:
        try:
            y0 = float(sp.N(derivada.subs(x, a)))
            m = float(sp.N(segunda_derivada.subs(x, a)))

            if not np.isfinite(y0) or not np.isfinite(m):
                raise ValueError

            y_tangente = y0 + m * (valores_x - a)

            texto = texto_ecuacion_punto_pendiente(a, y0, m)
            anotaciones.append(texto)

            fig.add_trace(go.Scatter(
                x=valores_x,
                y=y_tangente,
                mode="lines",
                name=f"Tangente, a={round(float(a), 3)}",
                line=dict(dash="dash"),
                visible=False,
                hovertemplate=texto + "<extra></extra>"
            ))

            fig.add_trace(go.Scatter(
                x=[a],
                y=[y0],
                mode="markers",
                name=f"Punto ({round(float(a), 3)}, {round(float(y0), 3)})",
                marker=dict(size=9),
                visible=False,
                hovertemplate=texto + "<extra></extra>"
            ))

        except:
            texto = (
                f"<b>Tangente no definida</b><br>"
                f"a = {formato_numero(a)}<br>"
                f"En este punto puede no existir f'(a) o f''(a)."
            )

            anotaciones.append(texto)

            fig.add_trace(go.Scatter(
                x=valores_x,
                y=np.full_like(valores_x, np.nan),
                mode="lines",
                name=f"Tangente no definida, a={round(float(a), 3)}",
                visible=False
            ))

            fig.add_trace(go.Scatter(
                x=[a],
                y=[np.nan],
                mode="markers",
                name=f"Punto no definido, a={round(float(a), 3)}",
                visible=False
            ))

    indice_inicial = len(valores_a) // 2

    fig.data[2 + 2 * indice_inicial].visible = True
    fig.data[2 + 2 * indice_inicial + 1].visible = True

    def visibilidad_para_indice(i, mostrar_fx=True, mostrar_fp=True):
        vis = [mostrar_fx, mostrar_fp] + [False] * (2 * len(valores_a))

        vis[2 + 2 * i] = True
        vis[2 + 2 * i + 1] = True

        return vis

    pasos_slider = []

    for i, a in enumerate(valores_a):
        try:
            y0 = float(sp.N(derivada.subs(x, a)))
            m = float(sp.N(segunda_derivada.subs(x, a)))

            titulo = (
                f"Gráficas interactivas | a = {round(float(a), 4)} | "
                f"f'(a) = {formato_numero(y0)}, f''(a) = {formato_numero(m)}"
            )

        except:
            titulo = f"Gráficas interactivas | Tangente no definida en a = {round(float(a), 4)}"

        paso = dict(
            method="update",
            args=[
                {"visible": visibilidad_para_indice(i, True, True)},
                {
                    "title": titulo,
                    "annotations": [
                        dict(
                            text=anotaciones[i],
                            xref="paper",
                            yref="paper",
                            x=0.01,
                            y=0.98,
                            showarrow=False,
                            align="left",
                            bordercolor="black",
                            borderwidth=1,
                            bgcolor="white",
                            opacity=0.90
                        )
                    ]
                }
            ],
            label=str(round(float(a), 2))
        )

        pasos_slider.append(paso)

    sliders = [
        dict(
            active=indice_inicial,
            currentvalue={
                "prefix": "Punto a = ",
                "font": {"size": 15}
            },
            pad={"t": 55},
            steps=pasos_slider
        )
    ]

    botones = [
        dict(
            label="Mostrar todo",
            method="update",
            args=[
                {"visible": visibilidad_para_indice(indice_inicial, True, True)}
            ]
        ),
        dict(
            label="Solo f(x)",
            method="update",
            args=[
                {
                    "visible": [True, False] + [False] * (2 * len(valores_a))
                }
            ]
        ),
        dict(
            label="Solo f'(x)",
            method="update",
            args=[
                {
                    "visible": [False, True] + [False] * (2 * len(valores_a))
                }
            ]
        ),
        dict(
            label="f'(x) + tangente",
            method="update",
            args=[
                {"visible": visibilidad_para_indice(indice_inicial, False, True)}
            ]
        )
    ]

    fig.add_hline(y=0, line_width=1)
    fig.add_vline(x=0, line_width=1)

    fig.update_layout(
        title="Gráficas interactivas: f(x), f'(x) y tangente sobre f'(x)",
        xaxis_title="x",
        yaxis_title="y",
        width=1000,
        height=700,
        hovermode="closest",
        dragmode="pan",
        sliders=sliders,
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                buttons=botones,
                x=0,
                y=1.18,
                showactive=True
            )
        ],
        annotations=[
            dict(
                text=anotaciones[indice_inicial],
                xref="paper",
                yref="paper",
                x=0.01,
                y=0.98,
                showarrow=False,
                align="left",
                bordercolor="black",
                borderwidth=1,
                bgcolor="white",
                opacity=0.90
            )
        ],
        legend=dict(
            title="Haz clic para activar/desactivar",
            orientation="h",
            yanchor="bottom",
            y=1.03,
            xanchor="left",
            x=0
        )
    )

    fig.update_xaxes(
        showgrid=True,
        zeroline=True,
        rangeslider_visible=False
    )

    fig.update_yaxes(
        showgrid=True,
        zeroline=True
    )

    display(Markdown("""
### Instrucciones de la gráfica

- Puedes hacer **zoom** con la rueda del mouse.
- Puedes **arrastrar** la gráfica para moverte por el plano.
- Puedes activar o desactivar curvas desde la **leyenda**.
- Usa el deslizador inferior para mover el punto `a`.
- Para cada punto `a`, el sistema calcula la recta tangente a `f'(x)` usando:

`L(x) = f'(a) + f''(a)(x-a)`

En el recuadro de la gráfica aparece también la forma punto-pendiente:

`y - f'(a) = f''(a)(x-a)`
"""))

    html = fig.to_html(include_plotlyjs="cdn")
    display(HTML(html))


print("Celda 4 ejecutada correctamente.")

Celda 4 ejecutada correctamente.


In [ ]:

# ============================================================
# CELDA 5
# PROGRAMA PRINCIPAL E INTERFAZ
# ============================================================

def calculadora_derivadas(funcion_usuario, xmin=-5, xmax=5):
    """
    Programa principal de la calculadora.
    """

    try:
        if xmin >= xmax:
            raise ValueError("El valor de xmin debe ser menor que xmax.")

        expr_original = leer_funcion(funcion_usuario)
        expr = sp.simplify(expr_original)

        derivada_sin_simplificar = derivada_no_simplificada(expr, x)
        derivada_simplificada = sp.factor(sp.simplify(derivada_sin_simplificar))

        segunda_derivada_sin_simplificar = derivada_no_simplificada(
            derivada_simplificada,
            x
        )

        segunda_derivada_simplificada = sp.factor(
            sp.simplify(segunda_derivada_sin_simplificar)
        )

        # ====================================================
        # SECCIÓN 1
        # ====================================================

        display(Markdown("# Calculadora de derivadas"))

        display(Markdown("## 1. Función ingresada"))

        display(Math(r"f(x)=" + sp.latex(expr_original)))

        display(Markdown("### Simplificación de la función"))

        if expr == expr_original:
            display(Math(r"\text{No se pudo simplificar más.}"))
        else:
            display(Math(r"f(x)=" + sp.latex(expr)))

        mostrar_raices_y_ceros(expr, xmin, xmax)

        # ====================================================
        # SECCIÓN 2
        # ====================================================

        display(Markdown("## 2. Derivada"))

        display(Markdown("### Derivada sin simplificar"))

        display(Markdown(
            "Esta es la expresión obtenida aplicando reglas de derivación antes de simplificar."
        ))

        display(Math(r"f'(x)=" + sp.latex(derivada_sin_simplificar)))

        display(Markdown("### Derivada simplificada"))

        display(Math(r"f'(x)=" + sp.latex(derivada_simplificada)))

        # ====================================================
        # SECCIÓN 3
        # ====================================================

        display(Markdown("## 3. Segunda derivada"))

        display(Markdown(
            "La segunda derivada se obtiene derivando la primera derivada."
        ))

        display(Math(
            r"f''(x)=\frac{d}{dx}\left["
            + sp.latex(derivada_simplificada)
            + r"\right]"
        ))

        display(Markdown("### Segunda derivada sin simplificar"))

        display(Math(r"f''(x)=" + sp.latex(segunda_derivada_sin_simplificar)))

        display(Markdown("### Segunda derivada simplificada"))

        display(Math(r"f''(x)=" + sp.latex(segunda_derivada_simplificada)))

        # ====================================================
        # SECCIÓN 4
        # ====================================================

        display(Markdown("## 4. Procedimiento robusto de derivación"))

        display(Markdown(
            "A continuación se descompone la función por capas internas: "
            "sumas, productos, cocientes, potencias y funciones compuestas."
        ))

        mostrar_procedimiento_derivada_robusto(expr, x)

        # ====================================================
        # SECCIÓN 5
        # ====================================================

        mostrar_analisis_dominio(
            expr,
            derivada_simplificada,
            segunda_derivada_simplificada,
            xmin,
            xmax
        )

        # ====================================================
        # SECCIÓN 6
        # ====================================================

        display(Markdown("## 6. Gráficas interactivas"))

        display(Markdown(
            "En esta sección aparecen juntas la función original, su derivada "
            "y la tangente deslizable sobre la derivada."
        ))

        display(Math(r"L(x)=f'(a)+f''(a)(x-a)"))

        display(Math(r"y-f'(a)=f''(a)(x-a)"))

        grafica_interactiva(
            expr,
            derivada_simplificada,
            segunda_derivada_simplificada,
            xmin,
            xmax
        )

    except Exception as error:
        display(Markdown("# Error"))

        print(error)
        print()
        print("Revisa que la función esté escrita correctamente.")
        print()
        print("Ejemplos válidos:")
        print("x^2 + 3*x - 5")
        print("x^3 - 3*x^2 + 2")
        print("sin(x)")
        print("cos(x^2)")
        print("ln(x)")
        print("sqrt(x + 1)")
        print("exp(x)")
        print("ln(sin(x^2 + 1))")
        print("arctan(ln(x))")
        print("e^x")


# ============================================================
# INTERFAZ DE ENTRADA
# ============================================================

entrada_funcion = widgets.Text(
    value="x^3 - 3*x^2 + 2",
    description="f(x):",
    layout=widgets.Layout(width="80%")
)

entrada_xmin = widgets.FloatText(
    value=-5,
    description="x mínimo:"
)

entrada_xmax = widgets.FloatText(
    value=5,
    description="x máximo:"
)

boton_calcular = widgets.Button(
    description="Calcular",
    button_style="success"
)


def mostrar_interfaz():
    """
    Muestra la interfaz principal.
    """

    display(Markdown("# Calculadora de derivadas en una variable"))

    display(Markdown(
        "Escribe una función usando solamente la variable `x`. "
        "No escribas `f(x)=`, solo la expresión."
    ))

    display(Markdown("""
### Ejemplos que puedes probar

`x^3 - 3*x^2 + 2`

`sin(x)`

`cos(x^2)`

`ln(x)`

`exp(x^2 + 1)`

`arctan(ln(x))`

`ln(sin(x^2 + 1))`

`e^x`
"""))

    display(entrada_funcion)

    display(widgets.HBox([
        entrada_xmin,
        entrada_xmax
    ]))

    display(boton_calcular)


def al_presionar_boton(boton):
    """
    Ejecuta la calculadora cuando se presiona el botón.
    """

    clear_output(wait=True)

    mostrar_interfaz()

    calculadora_derivadas(
        entrada_funcion.value,
        entrada_xmin.value,
        entrada_xmax.value
    )


boton_calcular.on_click(al_presionar_boton)

mostrar_interfaz()

# Calculadora de derivadas en una variable

Escribe una función usando solamente la variable `x`. No escribas `f(x)=`, solo la expresión.


### Ejemplos que puedes probar

`x^3 - 3*x^2 + 2`

`sin(x)`

`cos(x^2)`

`ln(x)`

`exp(x^2 + 1)`

`arctan(ln(x))`

`ln(sin(x^2 + 1))`

`e^x`


Text(value='sin(cos(tan(e^x^10)))', description='f(x):', layout=Layout(width='80%'))

Button(button_style='success', description='Calcular', style=ButtonStyle())

...

<IPython.core.display.Math object>

**Paso 4. Resolver la ecuación en los números reales**

**Regla usada:** Si no es polinómica, se intenta resolver simbólica y numéricamente.

**Resultado de raíces reales encontradas:**

<IPython.core.display.Math object>